In [ ]:
from google.colab import files
files.upload()   # upload kaggle.json


In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json


In [ ]:
!kaggle datasets download -d harshithabhaskaran/Echonet -p ./datasets/EchoNet --unzip


In [ ]:
!7z x ./datasets/EchoNet/EchoNet-Dynamic.7z.001 -o./Processed/EchoNet/


In [ ]:
from google.colab import files

uploaded = files.upload()  # Select frames.zip and masks.zip


In [ ]:
import zipfile
import os

# Create folders to extract
frames_folder = '/content/frames'
masks_folder = '/content/masks'
os.makedirs(frames_folder, exist_ok=True)
os.makedirs(masks_folder, exist_ok=True)

# Unzip frames
with zipfile.ZipFile('frames.zip', 'r') as zip_ref:
    zip_ref.extractall(frames_folder)

# Unzip masks
with zipfile.ZipFile('masks.zip', 'r') as zip_ref:
    zip_ref.extractall(masks_folder)

print(f"Frames extracted: {len(os.listdir(frames_folder))}")
print(f"Masks extracted: {len(os.listdir(masks_folder))}")


In [ ]:
!ls {frames_folder} | head
!ls {masks_folder} | head


In [ ]:
import os
import shutil

# Paths
frames_dir = "/content/frames"  # folder containing all frames
masks_dir = "/content/masks"    # folder containing all masks
output_frames = "frames_clean"
output_masks = "masks_clean"

# Create output folders if they don't exist
os.makedirs(output_frames, exist_ok=True)
os.makedirs(output_masks, exist_ok=True)

# Get list of frame filenames (without extensions for matching)
frame_files = {os.path.splitext(f)[0]: f for f in os.listdir(frames_dir)}
mask_files = {os.path.splitext(f)[0]: f for f in os.listdir(masks_dir)}

# Keep only masks that have corresponding frames
common_keys = set(frame_files.keys()) & set(mask_files.keys())
print(f"Frames before: {len(frame_files)}, Masks before: {len(mask_files)}")
print(f"Aligned pairs found: {len(common_keys)}")

# Copy aligned frames and masks to new folders
for key in common_keys:
    shutil.copy(os.path.join(frames_dir, frame_files[key]), os.path.join(output_frames, frame_files[key]))
    shutil.copy(os.path.join(masks_dir, mask_files[key]), os.path.join(output_masks, mask_files[key]))

print("✅ Done! Aligned frames and masks are ready in:", output_frames, "and", output_masks)


<a id='1'></a>
<h1 style='background: #87CEEB;padding-top: 20px; border:0; color:midnightblue'><center>***Import and Install Python Libraries***</center></h1>

In [ ]:
! pip install -U albumentations
! pip install opencv-python
! pip install segmentation-models-pytorch
! pip install pillow
! pip install pandas
! pip install matplotlib

In [ ]:
# Standard libraries
import os
import random
from glob import glob

# Numerical and image processing
import numpy as np
import pandas as pd
import cv2
from PIL import Image
import matplotlib.pyplot as plt
from scipy.ndimage import binary_dilation

# PyTorch and related modules
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.transforms import Grayscale
import torchvision.models.segmentation as tv_models

# Albumentations
import albumentations as A
from albumentations import (
    Compose, Resize, RandomRotate90, HorizontalFlip, VerticalFlip,
    RandomBrightnessContrast, ElasticTransform, GridDistortion,
    Normalize, ToGray, GaussNoise, RandomResizedCrop, Rotate
)
from albumentations.pytorch import ToTensorV2

# Segmentation Models PyTorch
import segmentation_models_pytorch as smp
from segmentation_models_pytorch.losses import DiceLoss, FocalLoss

# Progress bar
from tqdm import tqdm


# Device configuration

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

<a id='2'></a>
<h1 style='background: #87CEEB;padding-top: 20px; border:0; color:midnightblue'><center>***Load Dataset***</center></h1>

# Set address based on environment

In [ ]:
# -----------------------------
# Paths Setup
# -----------------------------
kaggle = False  # True if running in Kaggle, False if local/Colab

if kaggle:
    # Kaggle dataset paths
    masks_path = '/kaggle/input/echonet-dynamic-gt-image-frames/echonet_Dynamic_gt_masks/masks'
    images_path = '/kaggle/input/echonet-dynamic-gt-image-frames/echonet_Dynamic_gt_image_frames/frames'
    test_train_val_file_path = '/kaggle/input/echonet-dynamic/EchoNet-Dynamic/FileList.csv'
    csv_path_mask = '/kaggle/input/echonet-dynamic/EchoNet-Dynamic/VolumeTracings.csv'
else:
    # Local/Colab paths (after uploading and unzipping)
    base_path = '/content'  # change if your base folder is different
    masks_path = f'{base_path}/masks_clean'   # Use aligned masks folder
    images_path = f'{base_path}/frames_clean' # Use aligned frames folder
    test_train_val_file_path = f'{base_path}/Processed/EchoNet/EchoNet-Dynamic/FileList.csv'
    csv_path_mask = f'{base_path}/Processed/EchoNet/EchoNet-Dynamic/VolumeTracings.csv'

# Verify paths
print("✅ Masks path:", masks_path)
print("✅ Images path:", images_path)
print("✅ FileList CSV:", test_train_val_file_path)
print("✅ VolumeTracings CSV:", csv_path_mask)


# Image and Masks

In [ ]:
def plot_combined_mask_overlay(image_folder, mask_folder, count=8, seed=42, width=112, height=112):
    """
    Plot `count` samples with [Original Image, Mask, Overlay].
    Reads images from `image_folder` and masks from `mask_folder`.
    Assumes mask filenames match image filenames, e.g., `frame.png`.

    Displays:
    - Original image
    - Binary mask
    - Overlay with neon-orange border and 25% mask fill.
    """
    # Get sorted matching image-mask filenames
    image_files = sorted([
        f for f in os.listdir(image_folder) if f.endswith('.png')
    ])
    mask_files = sorted([
        f for f in os.listdir(mask_folder) if f.endswith('.png') and f in image_files
    ])

    # Check for matches
    matched_files = list(set(image_files).intersection(mask_files))
    if len(matched_files) < count:
        raise ValueError(f"Only {len(matched_files)} matching image-mask pairs found, but {count} requested.")

    # Select fixed random samples
    random.seed(seed)
    selected_files = random.sample(matched_files, count)

    # Plotting setup
    fig, axes = plt.subplots(count, 3, figsize=(12, 3 * count))
    if count == 1:
        axes = [axes]

    for i, fname in enumerate(selected_files):
        img_path = os.path.join(image_folder, fname)
        mask_path = os.path.join(mask_folder, fname)

        # Read image and mask
        img = cv2.imread(img_path)
        img_rgb = cv2.cvtColor(cv2.resize(img, (width, height)), cv2.COLOR_BGR2RGB)

        mask = Image.open(mask_path).resize((width, height))
        mask_np = np.array(mask)
        if mask_np.max() > 1:
            mask_np = (mask_np > 127).astype(np.uint8)

        # Mask border detection
        dilated = binary_dilation(mask_np, structure=np.ones((3, 3)))
        border = (dilated & ~mask_np).astype(np.uint8)

        # Overlay creation
        overlay = img_rgb.copy()
        neon_orange = np.array([255, 110, 0], dtype=np.uint8)

        # Inner fill: 25% neon orange
        for c in range(3):
            overlay[:, :, c] = np.where(
                mask_np == 1,
                (0.25 * neon_orange[c] + 0.65 * overlay[:, :, c]).astype(np.uint8),
                overlay[:, :, c]
            )

        # Border: full neon orange
        for c in range(3):
            overlay[:, :, c] = np.where(
                border == 1,
                neon_orange[c],
                overlay[:, :, c]
            )

        # Plot original, mask, and overlay
        axes[i][0].imshow(img_rgb)
        axes[i][0].set_title("Original Frame")
        axes[i][1].imshow(mask_np, cmap='gray')
        axes[i][1].set_title("Mask")
        axes[i][2].imshow(overlay)
        axes[i][2].set_title("Overlay")

        for ax in axes[i]:
            ax.axis('off')

    plt.tight_layout()
    plt.show()


In [ ]:
plot_combined_mask_overlay(images_path, masks_path, count=6, seed=42)

In [ ]:
df_train_test_val = pd.read_csv(test_train_val_file_path)
display(df_train_test_val)

In [ ]:
split_ratios = df_train_test_val['Split'].value_counts(normalize=True)
split_ratios_percent = split_ratios * 100

print("\nSplit ratios (as percentages):")
print(split_ratios_percent)

In [ ]:
class SegmentationDataset(Dataset):
    def __init__(self, image_paths, mask_paths, transform=None):
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.transform = transform


    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = cv2.imread(self.image_paths[idx])
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(self.mask_paths[idx], cv2.IMREAD_GRAYSCALE)

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image, mask = augmented['image'], augmented['mask']

        # Ensure mask is a PyTorch tensor before operations
        if not isinstance(mask, torch.Tensor):
            mask = torch.tensor(mask, dtype=torch.float32)

        mask = (mask > 0.5).float().unsqueeze(0)  # Binary mask
        return image, mask



In [ ]:
class EarlyStopping:
    def __init__(self, patience=5, delta=0.0):
        self.patience = patience
        self.counter = 0
        self.best_val_dice = None
        self.early_stop = False
        self.delta = delta

    def __call__(self, val_dice):
        if self.best_val_dice is None or val_dice > self.best_val_dice + self.delta:
            self.best_val_dice = val_dice
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True


In [ ]:
def get_transforms(train=True, mean=(0.0,), std=(1.0,)):  # Use dataset-specific mean/std
    if train:
        return Compose([
            Resize(512, 512),
            RandomBrightnessContrast(p=0.3),
            # A.GaussNoise(var_limit=(10, 50), p=0.5),
            A.MultiplicativeNoise(multiplier=(0.9, 1.1), p=0.3), # Add multiplicative noise
            ElasticTransform(alpha=1, sigma=50, p=0.3),
            GridDistortion(p=0.1),
            HorizontalFlip(p=0.5),  # Add horizontal flip
            Rotate(limit=10, p=0.3),
            Normalize(mean=mean, std=std), # Use dataset-specific mean/std
            ToTensorV2()
        ])
    else:
        return Compose([
            Resize(512, 512),
            Normalize(mean=mean, std=std), # Use dataset-specific mean/std
            ToTensorV2()
        ])

In [ ]:
# Function to gather image-mask pairs for a list of FileNames
def get_image_mask_pairs(file_names, images_path, masks_path):
    image_paths, mask_paths = [], []
    for name in file_names:
        # Find all images/masks starting with that name
        matching_images = sorted(glob(os.path.join(images_path, f"{name}_*.png")))
        matching_masks = sorted(glob(os.path.join(masks_path, f"{name}_*.png")))

        # Sanity check
        if len(matching_images) != len(matching_masks):
            print(f"Warning: {name} has {len(matching_images)} images and {len(matching_masks)} masks")

        image_paths.extend(matching_images)
        mask_paths.extend(matching_masks)

    return image_paths, mask_paths

In [ ]:
# Group FileNames by split
train_names = df_train_test_val[df_train_test_val['Split'] == 'TRAIN']['FileName'].tolist()
val_names = df_train_test_val[df_train_test_val['Split'] == 'VAL']['FileName'].tolist()
test_names = df_train_test_val[df_train_test_val['Split'] == 'TEST']['FileName'].tolist()

# Get image-mask pairs for each split
train_images, train_masks = get_image_mask_pairs(train_names, images_path, masks_path)
val_images, val_masks = get_image_mask_pairs(val_names, images_path, masks_path)
test_images, test_masks = get_image_mask_pairs(test_names, images_path, masks_path)

train_dataset = SegmentationDataset(train_images, train_masks, transform=get_transforms(train=True))
val_dataset = SegmentationDataset(val_images, val_masks, transform=get_transforms(train=False))
test_dataset = SegmentationDataset(test_images, test_masks, transform=get_transforms(train=False))

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

<a id='10'></a>
<h1 style='background: #87CEEB;padding-top: 20px; border:0; color:midnightblue'><center>***Model Definition***</center></h1>

In [ ]:
# class ResNet50UNet(nn.Module):
#     def __init__(self, num_classes=1, input_size=112):
#         super().__init__()
#         self.input_size = input_size
#         base_model = models.resnet50(weights="DEFAULT")

#         # Encoder
#         self.enc1 = nn.Sequential(base_model.conv1, base_model.bn1, base_model.relu)  # -> 56×56
#         self.enc2 = nn.Sequential(base_model.maxpool, base_model.layer1)              # -> 28×28
#         self.enc3 = base_model.layer2                                                 # -> 14×14
#         self.enc4 = base_model.layer3                                                 # -> 7×7
#         self.enc5 = base_model.layer4                                                 # -> 4×4

#         # Decoder (gradually upsample to 112x112)
#         self.up4 = self._up_block(2048, 1024)   # 4×4 → 8×8
#         self.up3 = self._up_block(1024, 512)    # 8×8 → 16×16
#         self.up2 = self._up_block(512, 256)     # 16×16 → 32×32
#         self.up1 = self._up_block(256, 64)      # 32×32 → 64×64
#         self.up0 = self._up_block(64, 32)       # 64×64 → 112×112

#         # Final output
#         self.final_conv = nn.Conv2d(32, num_classes, kernel_size=1)

#     def _up_block(self, in_channels, out_channels):
#         return nn.Sequential(
#             nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
#             nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
#             nn.BatchNorm2d(out_channels),
#             nn.ReLU(inplace=True)
#         )

#     def forward(self, x):
#         e1 = self.enc1(x)  # 56×56
#         e2 = self.enc2(e1) # 28×28
#         e3 = self.enc3(e2) # 14×14
#         e4 = self.enc4(e3) # 7×7
#         e5 = self.enc5(e4) # 4×4

#         d4 = self.up4(e5)            # 8×8
#         d3 = self.up3(d4 + e4)       # 16×16
#         d2 = self.up2(d3 + e3)       # 32×32
#         d1 = self.up1(d2 + e2)       # 64×64
#         d0 = self.up0(d1 + e1)       # 112×112

#         out = self.final_conv(d0)    # Final prediction map

#         return {'out': out}




model = smp.Unet(
    encoder_name="resnet50",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,  # 1 for binary segmentation
    activation=None  # Use raw logits (for BCEWithLogitsLoss)
)



# Initialize model
# model = ResNet50UNet()
model.to(device)
print("ResNet50UNet initialized.")


<a id='11'></a>
<h1 style='background: #87CEEB;padding-top: 20px; border:0; color:midnightblue'><center>***Training Parameter Definition ResNet50UNet***</center></h1>

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=0.00001)
criterion = nn.BCEWithLogitsLoss()
# criterion = FocalLoss('binary')
early_stopping = EarlyStopping(patience=10, delta=0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", patience=2, factor=0.5)

<a id='12'></a>
<h1 style='background: #87CEEB;padding-top: 20px; border:0; color:midnightblue'><center>***Training Parameter Definition Training Loop***</center></h1>

# Useful methods

In [ ]:
# IoU and Dice metrics
def calculate_iou(pred, target, smooth=1e-6):
    intersection = (pred * target).sum()
    union = pred.sum() + target.sum()
    return (intersection + smooth) / (union - intersection + smooth)

def calculate_dice(pred, target, smooth=1e-6):
    intersection = (pred * target).sum()
    return (2. * intersection + smooth) / (pred.sum() + target.sum() + smooth)

def evaluate_model(model, data_loader, device, criterion):
    model.eval()
    total_loss = 0.0
    total_iou = 0.0
    total_dice = 0.0

    with torch.no_grad():
        for images, masks in data_loader:
            images, masks = images.to(device), masks.to(device)
            outputs = model(images)

            if isinstance(outputs, dict):
                outputs = outputs['out']

            if masks.ndim == 3:
                masks = masks.unsqueeze(1)

            loss = criterion(outputs, masks)
            total_loss += loss.item()

            preds = (outputs > 0.5).float()

            total_iou += calculate_iou(preds, masks).item()
            total_dice += calculate_dice(preds, masks).item()

    avg_loss = total_loss / len(data_loader)
    avg_iou = total_iou / len(data_loader)
    avg_dice = total_dice / len(data_loader)

    return avg_loss, avg_iou, avg_dice

In [ ]:
# --- Save checkpoint function ---
def save_checkpoint(state, filename="best_model_checkpoint.pth"):
    torch.save(state, filename)
    print(f"✅ Checkpoint saved to {filename}")

# --- Initialize or Load Checkpoint ---
history = {'train_loss': [], 'val_loss': [], 'val_IoU': [], 'val_dice': []}
best_val_dice = 0.0
start_epoch = 0

checkpoint_path = "heart_ecohnet_dynamic_segmentation_ResNet50UNet_imagenet_checkpoint.pth"

if os.path.exists(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    start_epoch = checkpoint['epoch']
    best_val_dice = checkpoint.get('best_val_dice', 0.0)
    history = checkpoint.get('history', history)
    print(f"✅ Loaded checkpoint from epoch {start_epoch}, best val_dice = {best_val_dice:.4f}")
else:
    print("⚠️ No checkpoint found. Training will start from scratch.")


In [ ]:
# --- Save checkpoint function ---
def save_checkpoint(state, filename="best_model_checkpoint.pth"):
    torch.save(state, filename)
    print(f"✅ Checkpoint saved to {filename}")

# --- Initialize or Load Checkpoint ---
history = {'train_loss': [], 'val_loss': [], 'val_IoU': [], 'val_dice': []}
best_val_dice = 0.0
start_epoch = 0

# Set checkpoint path based on class-aware sampling
checkpoint_path = "heart_ecohnet_dynamic_segmentation_ResNet50UNet_imagenet_checkpoint.pth"

# Load checkpoint if it exists
if os.path.exists(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    start_epoch = checkpoint['epoch']
    best_val_dice = checkpoint.get('best_val_dice', 0.0)
    history = checkpoint.get('history', history)
    print(f"✅ Loaded checkpoint from epoch {start_epoch}, best val_dice = {best_val_dice:.4f}")


In [ ]:
def train_model(model, train_loader, val_loader, optimizer, criterion, scheduler, device, num_epochs=10, start_epoch=0):
    global best_val_dice, history  # needed to persist best_val_dice and history between function calls

    for epoch in range(start_epoch, num_epochs):
        model.train()
        train_loss = 0
        skipped_batches = 0

        for batch_idx, (images, masks) in enumerate(tqdm(train_loader, desc=f'Training Epoch {epoch + 1}/{num_epochs}')):
            if images.shape[0] <= 1:
                print(f"Skipping batch {batch_idx} with batch size {images.shape[0]}")
                skipped_batches += 1
                continue

            images, masks = images.to(device), masks.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, masks)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        # Validation
        val_loss, val_iou, val_dice = evaluate_model(model, val_loader, device, criterion)
        scheduler.step(val_loss)

        # Normalize train loss
        effective_batches = len(train_loader) - skipped_batches
        train_loss /= max(1, effective_batches)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_IoU'].append(val_iou)
        history['val_dice'].append(val_dice)

        print(f"Epoch {epoch + 1}: Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, IoU: {val_iou:.4f}, Dice: {val_dice:.4f}")

        # --- Save checkpoint if best dice improved ---
        if val_dice > best_val_dice:
            best_val_dice = val_dice
            save_checkpoint({
                'epoch': epoch + 1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_val_dice': best_val_dice,
                'history': history,
            }, filename=checkpoint_path)

        # --- Early stopping logic ---
        early_stopping(val_dice)

        if early_stopping.early_stop:
            print(f"🛑 Early stopping triggered after {epoch+1} epochs. Best val_dice = {early_stopping.best_val_dice:.4f}")
            break


# Run training
train_model(model, train_loader, val_loader, optimizer, criterion, scheduler, device, num_epochs=100,start_epoch=start_epoch)

<a id='13'></a>
<h1 style='background: #87CEEB;padding-top: 20px; border:0; color:midnightblue'><center>***Performance Visulization***</center></h1>

In [ ]:
plt.figure(figsize=(7, 7))
plt.plot(history['train_loss'], label='Training loss')
plt.plot(history['val_loss'], label='Validation loss')
plt.ylim(0, 0.4)
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(7, 7))
plt.plot(history['val_IoU'], label='Validation mean Jaccard index')
plt.plot(history['val_dice'], label='Validation Dice coefficient')
plt.legend()
plt.show()

<a id='4'></a>
<h1 style='background: #87CEEB;padding-top: 20px; border:0; color:midnightblue'><center>***Performance on Test ***</center></h1>

In [ ]:
def display_predictions(model, loader, device, num_images=5):
    model.eval()
    with torch.no_grad():

        images, masks = next(iter(loader))
        images, masks = images.to(device), masks.to(device)
        outputs = model(images)

        for i in range(num_images):
            plt.figure(figsize=(12, 4))

            plt.subplot(1, 3, 1)
            plt.imshow(images[i].permute(1, 2, 0).cpu().numpy())
            plt.title("Input Image")

            plt.subplot(1, 3, 2)
            plt.imshow(masks[i].squeeze().cpu().numpy(), cmap='gray')
            plt.title("True Mask")

            plt.subplot(1, 3, 3)
            plt.imshow(torch.sigmoid(outputs[i]).squeeze().detach().cpu().numpy(), cmap='gray')  # Apply sigmoid and detach.
            plt.title("Predicted Mask")

            plt.show()

display_predictions(model, val_loader, device)

# Evaluate on Test Data

In [ ]:
test_avg_loss, test_avg_iou, test_avg_dice = evaluate_model(model, test_loader, criterion, device)
print(f"Test Loss: {test_avg_loss:.4f}, Test IoU: {test_avg_iou:.4f}, Test dice: {test_avg_dice:.4f}" )

In [ ]:
def visualize_predictions(model, data_loader, device, num_images=10, mask_transparency=0.3):
    model.eval()  # Set the model to evaluation mode
    images_shown = 0

    with torch.no_grad():
        for batch_idx, (images, masks) in enumerate(data_loader):
            print(f"Batch index: {batch_idx}, images.shape: {images.shape if images is not None else None}, images_shown: {images_shown}") # Debugging print

            if images.numel() == 0 or masks.numel() == 0:
                print("Warning: Skipping empty visualization batch.")
                continue  # Skip to the next batch

            images = images.to(device)
            masks = masks.to(device)

            output = model(images)

            output = torch.sigmoid(output)
            outputs = (output > 0.5).float()  # Threshold predictions

            for i in range(images.size(0)):
                if images_shown >= num_images:
                    print("Reached num_images limit, breaking inner loop.") # Debugging print
                    break

                img = images[i].cpu().numpy().transpose((1, 2, 0))
                mask = masks[i].cpu().numpy().squeeze()
                pred_mask = outputs[i].cpu().numpy().squeeze()

                img = np.clip(img, 0, 1)

                fig, axes = plt.subplots(1, 4, figsize=(12, 4))  # Adjusted figure size for better visualization

                axes[0].imshow(img)
                axes[0].set_title("Original Image")
                axes[0].axis('off')

                axes[1].imshow(mask, cmap='gray')
                axes[1].set_title("Ground Truth Mask")
                axes[1].axis('off')

                axes[2].imshow(pred_mask, cmap='gray')
                axes[2].set_title("Predicted Mask")
                axes[2].axis('off')

                # Create the combined image for axes[3]
                combined_img = img.copy()

                # Apply ground truth mask (red)
                red_mask = np.zeros_like(img)
                red_mask[:, :, 0] = 1  # Red channel
                combined_img[mask == 1] = ((1 - mask_transparency) * combined_img[mask == 1] + mask_transparency * red_mask[mask == 1])


                # Apply predicted mask (neon green)
                neon_green_mask = np.zeros_like(img)
                neon_green_mask[:, :, 1] = 1  # Green channel
                combined_img[pred_mask == 1] = ((1 - mask_transparency) * combined_img[pred_mask == 1] + mask_transparency * neon_green_mask[pred_mask == 1])

                # Find contours of the masks
                mask_contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                pred_mask_contours, _ = cv2.findContours(pred_mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

                # Draw contours on the combined image.  Use a color that corresponds to the mask.
                cv2.drawContours(combined_img, mask_contours, -1, (1, 0, 0), 2)  # Red (BGR) for ground truth.
                cv2.drawContours(combined_img, pred_mask_contours, -1, (0, 1, 0), 2) # Neon Green (BGR) for prediction.



                axes[3].imshow(combined_img)
                axes[3].set_title("Combined Prediction")
                axes[3].axis('off')

                plt.tight_layout()
                plt.show()
                images_shown += 1
                print(f"Image {images_shown} shown.")

            if images_shown >= num_images:
                print("Reached num_images limit, breaking outer loop.") # Debugging print
                break

In [ ]:
visualize_predictions(model, test_loader, device, 10,0.1)

<a id='15'></a>
<h1 style='background: #87CEEB;padding-top: 20px; border:0; color:midnightblue'><center>***Save Model***</center></h1>

In [ ]:
# Save Model
torch.save(model.state_dict(), 'heart_ecohnet_dynamic_segmentation_ResNet50UNet_imagenet.pth')
print("ResNet50UNet Model saved successfully!")
